# Build modeling dataset V2

Add the frozen previous-loss history and exact one-year financial deltas before creating the t → t+1 target.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

df = pd.read_csv('../data/processed/financial_sporting_data.csv')
modeling = df.copy()

## Check club-year key

In [2]:
duplicates = modeling[
    modeling.duplicated(subset=['club', 'fiscal_year'], keep=False)
]

assert duplicates.empty
duplicates

,club,fiscal_year,cash,current_assets,total_assets,current_liabilities,total_liabilities,equity,net_result,division,...,league_size,promoted,relegated,table_source,table_source_url,outcome_source,notes,points_per_game,goal_difference_per_game,relative_league_position


## Create current-year financial features

In [3]:
modeling['current_liquidity'] = (
    modeling['current_assets']
    / modeling['current_liabilities'].replace(0, np.nan)
)

modeling['liability_ratio'] = (
    modeling['total_liabilities']
    / modeling['total_assets'].replace(0, np.nan)
)

modeling['return_on_assets'] = (
    modeling['net_result']
    / modeling['total_assets'].replace(0, np.nan)
)

modeling['loss_current_year'] = (modeling['net_result'] < 0).astype(int)

## Create previous loss rate

Use only observations preceding each club-year, following the frozen framework definition.

In [4]:
modeling = modeling.sort_values(['club', 'fiscal_year']).copy()

modeling['previous_loss_rate'] = (
    modeling.groupby('club')['net_result']
    .transform(lambda x: x.lt(0).shift().expanding().mean())
)

## Create exact one-year financial deltas

A delta is available only when the same club has an observation for the immediately preceding fiscal year.

In [5]:
previous_year = modeling[[
    'club',
    'fiscal_year',
    'current_liquidity',
    'liability_ratio',
    'return_on_assets',
]].copy()

previous_year = previous_year.rename(columns={
    'fiscal_year': 'previous_year',
    'current_liquidity': 'previous_current_liquidity',
    'liability_ratio': 'previous_liability_ratio',
    'return_on_assets': 'previous_return_on_assets',
})

previous_year['fiscal_year'] = previous_year['previous_year'] + 1

modeling = modeling.merge(
    previous_year,
    how='left',
    on=['club', 'fiscal_year'],
    validate='one_to_one',
)

modeling['delta_current_liquidity'] = (
    modeling['current_liquidity'] - modeling['previous_current_liquidity']
)
modeling['delta_liability_ratio'] = (
    modeling['liability_ratio'] - modeling['previous_liability_ratio']
)
modeling['delta_return_on_assets'] = (
    modeling['return_on_assets'] - modeling['previous_return_on_assets']
)

In [6]:
delta_available = modeling[[
    'delta_current_liquidity',
    'delta_liability_ratio',
    'delta_return_on_assets',
]].notna().any(axis=1)

assert (
    modeling.loc[delta_available, 'previous_year']
    == modeling.loc[delta_available, 'fiscal_year'] - 1
).all()

## Create exact t → t+1 target

In [7]:
next_year = df[['club', 'fiscal_year', 'net_result']].copy()

next_year = next_year.rename(columns={
    'fiscal_year': 'target_year',
    'net_result': 'net_result_next_year',
})

next_year['fiscal_year'] = next_year['target_year'] - 1

modeling = modeling.merge(
    next_year,
    how='left',
    on=['club', 'fiscal_year'],
    validate='one_to_one',
)

available_target = modeling['net_result_next_year'].notna()
modeling['loss_next_year'] = pd.NA
modeling.loc[available_target, 'loss_next_year'] = (
    modeling.loc[available_target, 'net_result_next_year'] < 0
).astype(int)
modeling['loss_next_year'] = modeling['loss_next_year'].astype('Int64')

## Audit unavailable targets

In [8]:
without_target = (
    modeling.loc[
        modeling['net_result_next_year'].isna(),
        ['club', 'fiscal_year'],
    ]
    .sort_values(['fiscal_year', 'club'])
    .reset_index(drop=True)
)

without_target[
    without_target['fiscal_year'] < modeling['fiscal_year'].max()
]

,club,fiscal_year
0,Vitória,2016
1,Mirassol,2018
2,Figueirense,2023
3,CSA,2024


## Keep labeled one-year transitions

In [9]:
modeling = modeling[
    modeling['net_result_next_year'].notna()
].copy()

modeling['loss_next_year'] = modeling['loss_next_year'].astype(int)

assert (modeling['target_year'] == modeling['fiscal_year'] + 1).all()
assert modeling['loss_next_year'].isin([0, 1]).all()
assert not modeling.duplicated(subset=['club', 'fiscal_year']).any()

modeling.shape

(302, 42)

## V2 feature availability

In [10]:
financial_v2_features = [
    'current_liquidity',
    'liability_ratio',
    'return_on_assets',
    'loss_current_year',
    'previous_loss_rate',
    'delta_current_liquidity',
    'delta_liability_ratio',
    'delta_return_on_assets',
]

sporting_features = [
    'division',
    'points_per_game',
    'goal_difference_per_game',
    'relative_league_position',
    'promoted',
    'relegated',
]

financial_v2_complete = modeling[financial_v2_features].notna().all(axis=1)
sporting_complete = modeling[sporting_features].notna().all(axis=1)
common_v2 = financial_v2_complete & sporting_complete

pd.Series({
    'all_labeled_observations': len(modeling),
    'financial_v2_complete': int(financial_v2_complete.sum()),
    'common_financial_sporting_v2': int(common_v2.sum()),
})

all_labeled_observations        302
financial_v2_complete           266
common_financial_sporting_v2    262
dtype: int64

In [11]:
availability_by_year = pd.concat([
    modeling.groupby('fiscal_year').size().rename('labeled'),
    modeling.loc[financial_v2_complete].groupby('fiscal_year').size().rename('financial_v2'),
    modeling.loc[common_v2].groupby('fiscal_year').size().rename('common_v2'),
], axis=1).fillna(0).astype(int)

availability_by_year

,labeled,financial_v2,common_v2
fiscal_year,,,
2016,32,0,0
2017,33,32,31
2018,32,32,32
2019,33,32,32
2020,35,33,33
2021,35,35,35
2022,35,35,35
2023,34,34,33
2024,33,33,31


## Audit rows without complete V2 financial history

In [12]:
modeling.loc[
    ~financial_v2_complete,
    [
        'club',
        'fiscal_year',
        'previous_loss_rate',
        'delta_current_liquidity',
        'delta_liability_ratio',
        'delta_return_on_assets',
    ],
].sort_values(['fiscal_year', 'club']).reset_index(drop=True)

,club,fiscal_year,previous_loss_rate,delta_current_liquidity,delta_liability_ratio,delta_return_on_assets
0,América-MG,2016,NaN,NaN,NaN,NaN
1,Athletico-PR,2016,NaN,NaN,NaN,NaN
2,Atlético-GO,2016,NaN,NaN,NaN,NaN
3,Atlético-MG,2016,NaN,NaN,NaN,NaN
4,Avaí,2016,NaN,NaN,NaN,NaN
5,Bahia,2016,NaN,NaN,NaN,NaN
6,Botafogo,2016,NaN,NaN,NaN,NaN
7,CSA,2016,NaN,NaN,NaN,NaN
8,Ceará,2016,NaN,NaN,NaN,NaN
9,Chapecoense,2016,NaN,NaN,NaN,NaN


## Audit sporting exclusions from the V2 common sample

In [13]:
modeling.loc[
    financial_v2_complete & ~sporting_complete,
    ['club', 'fiscal_year'] + sporting_features,
].sort_values(['fiscal_year', 'club']).reset_index(drop=True)

,club,fiscal_year,division,points_per_game,goal_difference_per_game,relative_league_position,promoted,relegated
0,Mirassol,2017,NaN,NaN,NaN,NaN,NaN,NaN
1,Paraná Clube,2023,NaN,NaN,NaN,NaN,NaN,NaN
2,Paraná Clube,2024,NaN,NaN,NaN,NaN,NaN,NaN
3,Santa Cruz,2024,NaN,NaN,NaN,NaN,NaN,NaN


## Target distribution in the V2 common sample

In [14]:
modeling.loc[common_v2].groupby('fiscal_year')['loss_next_year'].agg(
    observations='count',
    losses='sum',
    loss_rate='mean',
)

,observations,losses,loss_rate
fiscal_year,,,
2017,31,16,0.516129
2018,32,17,0.531250
2019,32,23,0.718750
2020,33,15,0.454545
2021,35,15,0.428571
2022,35,14,0.400000
2023,33,24,0.727273
2024,31,20,0.645161


## 2020 sensitivity checkpoint

In [15]:
pd.Series({
    'common_v2_main': int(common_v2.sum()),
    'common_v2_without_feature_year_2020': int((common_v2 & (modeling['fiscal_year'] != 2020)).sum()),
})

common_v2_main                         262
common_v2_without_feature_year_2020    229
dtype: int64

## Export dataset

Keep all labeled transitions. The modeling notebook will apply the V2 common-sample filter.

In [ ]:
output_path = Path('../data/processed/modeling_data_v2.csv')
output_path.parent.mkdir(parents=True, exist_ok=True)

modeling.to_csv(output_path, index=False)

(302, 42)